# Dental Vision V1 — DENTEX training
T4 GPU workflow: clone, install, download, verify, extract, inspect, then train after resolving the real COCO paths.


In [5]:
!nvidia-smi


/bin/bash: line 1: nvidia-smi: command not found


In [6]:
import os
if not os.path.exists('/content/dental-vision-v1'):
    !git clone https://github.com/drhaidarali95/dental-vision-v1.git /content/dental-vision-v1
%cd /content/dental-vision-v1
!git pull
!pip -q install -r requirements.txt


/content/dental-vision-v1
Already up to date.


In [7]:
!python scripts/download_dentex.py --out data/dentex


Title: DENTEX CHALLENGE 2023
License: {'id': 'cc-by-4.0'}
Available files: training_data.zip, validation_data.zip
Traceback (most recent call last):
  File "/content/dental-vision-v1/scripts/download_dentex.py", line 56, in <module>
    main()
  File "/content/dental-vision-v1/scripts/download_dentex.py", line 53, in main
    download(url, pathlib.Path(args.out) / name, md5)
  File "/content/dental-vision-v1/scripts/download_dentex.py", line 21, in download
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f:
  File "/usr/lib/python3.13/urllib/request.py", line 189, in urlopen
    return opener.open(url, data, timeout)
  File "/usr/lib/python3.13/urllib/request.py", line 489, in open
    response = self._open(req, data)
  File "/usr/lib/python3.13/urllib/request.py", line 506, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
  File "/usr/lib/python3.13/urllib/request.py", line 466, in _call_chain
    result = func(*args)
  File "/usr/lib/python

In [8]:
import pathlib, zipfile
root = pathlib.Path('data/dentex')
for z in root.glob('*.zip'):
    dest = root / z.stem
    dest.mkdir(parents=True, exist_ok=True)
    marker = dest / '.extracted'
    if not marker.exists():
        print('Extracting', z, '->', dest)
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
        marker.touch()
    else:
        print('Already extracted:', z)
!python scripts/inspect_dentex.py data/dentex


Extracting data/dentex/training_data.zip -> data/dentex/training_data


BadZipFile: File is not a zip file

In [ ]:
import json, pathlib
root = pathlib.Path('data/dentex')
candidates=[]
for p in root.rglob('*.json'):
    try:
        d=json.loads(p.read_text())
    except Exception:
        continue
    if isinstance(d,dict) and {'images','annotations','categories'}.issubset(d):
        candidates.append((p,len(d['images']),len(d['annotations']),d['categories']))
for p,n,a,c in candidates:
    print('COCO:',p,'images=',n,'annotations=',a)
    print(' categories=',[(x.get('id'),x.get('name')) for x in c])
assert candidates, 'No COCO annotation JSON found after extraction.'


## Training
Do not guess paths. Use the COCO path printed above. The next cell resolves its image directory from COCO file_name entries and launches the baseline.


In [ ]:
import json, pathlib, subprocess, sys
ann = max(candidates, key=lambda x: x[1])[0]
d=json.loads(ann.read_text())
sample=d['images'][0]['file_name']
matches=list(root.rglob(pathlib.Path(sample).name))
assert matches, f'Could not locate sample image {sample}'
img_path=matches[0]
sample_parts=pathlib.Path(sample).parts
image_root=img_path
for _ in sample_parts:
    image_root=image_root.parent
print('Annotations:',ann)
print('Image root:',image_root)
cmd=[sys.executable,'train.py','--images',str(image_root),'--annotations',str(ann),'--epochs','20','--batch-size','2','--output','checkpoints']
print('Launching:', ' '.join(cmd))
subprocess.run(cmd, check=True)
